[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter19_saving_loading_and_running_on_a_gpu/notebook.ipynb)

# Chapter 19: Saving, Loading and Running on a GPU

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Every network you have trained in this book vanished the moment your Python session ended. The irises of Chapter 11, the digit reader of Chapter 17, the LSTM that learned to remember in Chapter 18: each one existed only as numbers in memory, and closing the notebook threw them away. Real work cannot run like that. A model that took a day to train has to survive the night, and a model you want other people to use has to leave your machine.

The second half of this chapter is about speed, and it begins with a story about hardware. In 2009 Rajat Raina, Anand Madhavan and Andrew Ng at Stanford reported that by running deep network training on the graphics cards built for video games, they could train up to seventy times faster than on an ordinary dual core processor. Three years later Alex Krizhevsky trained the network that opened Chapter 1 on two consumer graphics cards with three gigabytes of memory each. The network had sixty million weights, was too large to fit on one card, and so was split in half across the two, and it still took five to six days. Without those cards it would not have been tried at all.

This chapter covers both halves of keeping a model useful. Before you read on, try to guess the answers to these four questions.

1. **What exactly needs to be written to disk to save a trained network?**
2. **If training stops halfway, what do you need to carry on as if it never stopped?**
3. **Why would a chip designed for video games be good at training networks?**
4. **If you train the same network twice, should you get the same answer?**

## Checkpoints, Saving a Trained Model

Use the flowers once more, set up exactly as in Chapter 11.

In [ ]:
import os
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

The split and scaling are Chapter 11's, unchanged. Keep an eye on `mean` and `std`, because by the end of the chapter they will turn out to be part of the model.

In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3,
    random_state=7, stratify=iris.target)
mean, std = X_train.mean(axis=0), X_train.std(axis=0)
Xtr = torch.tensor((X_train - mean) / std, dtype=torch.float32)
Xte = torch.tensor((X_test - mean) / std, dtype=torch.float32)
ytr, yte = torch.tensor(y_train), torch.tensor(y_test)
loss_fn = nn.CrossEntropyLoss()

To keep every number exactly reproducible, this chapter trains on the whole training half at once with Adam, so there is no shuffling to account for.

In [ ]:
def make_net(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))

def train_steps(net, opt, n):
    for step in range(n):
        opt.zero_grad()
        loss = loss_fn(net(Xtr), ytr)
        loss.backward()
        opt.step()
    return loss.item()

net = make_net()
opt = torch.optim.Adam(net.parameters(), lr=0.01)
print(round(train_steps(net, opt, 100), 6))

A network's learned knowledge is entirely in its weights, and Chapter 10 showed where to find them. `state_dict()` returns them as a dictionary from names to tensors, and `torch.save` writes any such dictionary to a file.

In [ ]:
print(list(net.state_dict().keys()))
torch.save(net.state_dict(), "iris_weights.pt")
print(os.path.getsize("iris_weights.pt"), "bytes")

Four entries: the weight matrix and bias of each of the two linear layers, 131 numbers in all, exactly Chapter 11's count. The file is a few kilobytes. Notice what is *not* in it. The file holds no description of the network's shape, only its numbers. To use it again you build an empty network of the same shape in code and pour the numbers in, which is why the code that defines a model is as much a part of the saved model as the file.

That is enough for a finished model. For a training run that might be interrupted, it is not. Train for 50 steps, save, and try to carry on.

In [ ]:
net = make_net()
opt = torch.optim.Adam(net.parameters(), lr=0.01)
train_steps(net, opt, 50)
torch.save({"step": 50,
            "model": net.state_dict(),
            "optimizer": opt.state_dict(),
            "mean": mean.tolist(),
            "std": std.tolist()}, "iris_checkpoint.pt")
print(os.path.getsize("iris_checkpoint.pt"), "bytes")

A **checkpoint** is a dictionary of everything needed to pick up where you left off. This one holds the step count, the weights, the optimizer's own state, and the mean and spread used to scale the inputs. The file is more than twice the size of the weights alone, and the difference is the optimizer: Adam keeps two running averages for every single weight, as Chapter 13 explained, and those averages are part of where training stands.

> 🖼️ **Figure 19.1:** What goes into a training checkpoint. Only the top box is needed to make predictions, but leaving out any of the others costs you something later.

Now the test that matters. Build a fresh network with a deliberately different seed, so nothing is shared by accident, load the checkpoint into it, train 50 more steps, and compare with the uninterrupted run of 100.

In [ ]:
checkpoint = torch.load("iris_checkpoint.pt")

resumed = make_net(seed=99)
resumed_opt = torch.optim.Adam(resumed.parameters(), lr=0.01)
resumed.load_state_dict(checkpoint["model"])
resumed_opt.load_state_dict(checkpoint["optimizer"])
print("resumed", round(train_steps(resumed, resumed_opt, 50), 6))

forgetful = make_net(seed=99)
forgetful_opt = torch.optim.Adam(forgetful.parameters(), lr=0.01)
forgetful.load_state_dict(checkpoint["model"])
print("weights only", round(train_steps(forgetful, forgetful_opt, 50), 6))

The resumed run landed on 0.07926, the same as the uninterrupted run to the sixth decimal place, so stopping and starting made no difference at all. The run that reloaded only the weights ended somewhere else entirely. Its loss happens to be lower, but that is beside the point: its Adam started again from scratch with no memory of recent slopes, so it took different steps and is a different run. You could not reproduce it, and if a schedule from Chapter 14 depended on the step count, it would be wrong as well.

## Loading for Inference

**Inference** means using a trained model to make predictions rather than to learn. Three habits from earlier chapters come together here, and forgetting any one of them gives wrong answers without any error.

In [ ]:
model = make_net(seed=123)
model.load_state_dict(checkpoint["model"])
model.eval()

new_flower = [5.9, 3.0, 5.1, 1.8]
scaled = (torch.tensor(new_flower) - torch.tensor(checkpoint["mean"])) \
         / torch.tensor(checkpoint["std"])
with torch.no_grad():
    probs = torch.softmax(model(scaled.float().unsqueeze(0)), dim=1)[0]
print([round(p, 3) for p in probs.tolist()])
print(iris.target_names[probs.argmax().item()])

The network calls it a virginica at 0.712, but gives versicolor 0.282: this flower sits near the border between the two species that Chapter 11's scatter plot showed, and the network is right to hesitate. Three habits made that prediction trustworthy.

First, `model.eval()`, because dropout and batch normalization from Chapters 15 and 16 behave differently in training. This network has neither, but the habit costs nothing and the bug it prevents is silent. Second, `torch.no_grad()`, from Chapter 9, because there is no backward pass and no reason to record one. Third, and easiest to forget, the new measurements are scaled with the *training* mean and spread stored in the checkpoint. A flower fed in raw, in centimetres, would land somewhere the network has never seen, and Chapter 14 showed what unscaled inputs do. This is why the scaling numbers went into the checkpoint.

The `unsqueeze(0)` turns one flower into a batch of one, because layers expect a batch dimension. And `softmax` turns the three raw scores into probabilities, which the network itself never computes because `CrossEntropyLoss` does that inside the loss, as Chapter 11 explained.

One caution about the alternative you will see in older code: `torch.save(net)`, which pickles the whole Python object rather than its weights. It works, but the file then depends on your exact class definitions and folder layout, and loading a file like that from someone else can run code they put in it. Recent versions of PyTorch load only plain data by default for exactly that reason. Save the `state_dict`, keep the code that builds the network, and treat model files from strangers with the same suspicion as any other program.

## Moving to the GPU

A graphics card earns its speed from a different design. A processor has a handful of powerful cores, each able to run complicated, branching code very quickly. A graphics card has thousands of much simpler cores, each doing the same small arithmetic on different numbers at the same moment, because that is what drawing millions of pixels requires. Raina and his colleagues noticed that the multiplications inside a network have exactly that shape: the same multiply and add, repeated across huge matrices.

> 🖼️ **Figure 19.2:** Why a graphics card suits a network. Each green core is far weaker than a processor core, but a matrix multiplication splits into thousands of identical small jobs, and the card runs them all at once.

In PyTorch every tensor lives on a **device**, and arithmetic only happens between tensors on the same device. Moving to a graphics card is mostly a matter of choosing the device once and sending the model and every batch there.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

`cuda` is an NVIDIA card, `mps` is the graphics hardware in Apple silicon Macs, and `cpu` is the fallback. This book was checked on a machine without a graphics card, so the cell prints `cpu`; on yours it may not. Every cell in this book is written to run either way, and that is the point of choosing the device in one place.

In [ ]:
model = make_net().to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xtr_d, ytr_d = Xtr.to(device), ytr.to(device)
for step in range(100):
    opt.zero_grad()
    loss = loss_fn(model(Xtr_d), ytr_d)
    loss.backward()
    opt.step()
print(next(model.parameters()).device.type, round(loss.item(), 4))

The loss is 0.0793, the first run's 0.07926 rounded, because nothing changed but where the arithmetic happens. Three rules cover almost every device bug. Create the optimizer *after* moving the model, so it holds the moved weights. Move every batch inside the training loop with `xb.to(device)`, since a `DataLoader` hands out tensors on the processor. And bring results back with `.cpu()` before turning them into numpy arrays or saving them, since numpy knows nothing about graphics cards. Break any of these and PyTorch stops with an error that mentions two devices, which at least is not silent.

Do not expect a graphics card to speed up the flowers. Sending a few hundred numbers to the card and back costs more than the arithmetic, and a network with 131 weights has nothing to parallelise. The speedup appears when the matrices are large: wide layers, big batches, convolutions over real photographs, which is exactly the work Books 7 and 8 do. A checkpoint saved on a graphics card loads onto a machine without one by passing `map_location="cpu"` to `torch.load`.

## Reproducibility and Seeds

Every number in this book came out the same each time it was run, and that was not luck. Each experiment set `torch.manual_seed` before creating a network, and gave every `DataLoader` a seeded generator. Check that a seed really pins the starting weights.

In [ ]:
a, b, c = make_net(seed=0), make_net(seed=0), make_net(seed=1)
print(torch.equal(a[0].weight, b[0].weight))
print(torch.equal(a[0].weight, c[0].weight))

Same seed, identical weights; different seed, different weights. Now the more important question. If the seed decides the starting point, how much does the result depend on it?

In [ ]:
scores = []
for seed in range(10):
    net = make_net(seed)
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    train_steps(net, opt, 100)
    with torch.no_grad():
        scores.append(round((net(Xte).argmax(1) == yte).float().mean().item(), 3))
print(scores)
print(round(sum(scores) / len(scores), 3), min(scores), max(scores))

Ten seeds, two different answers: 0.978 or 0.956, which on 45 test flowers is the difference between one mistake and two. A report that said "the model scores 0.978" would be true of seven of these runs and false of three. The honest report is the average over seeds with its range, here 0.971 from 0.956 to 0.978, and the same rule applies to every comparison in this book. Chapters 17 and 18 checked their key results on several seeds for exactly this reason, and Chapter 17 found that one comparison, the transfer learning result, did not survive the check.

There are limits to how far seeds go. A run on a graphics card can differ in the last decimal places from the same run on a processor, because the card adds numbers in a different order and floating point addition is not perfectly associative. PyTorch can enforce identical results with `torch.use_deterministic_algorithms(True)`, at some cost in speed. For most work, the right goal is not identical digits everywhere but results whose conclusions do not change from seed to seed.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Load into the wrong shape.** Build a network with 8 hidden units instead of 16 and try to load `iris_weights.pt` into it. Read the error and explain it.
2. **Forget the scaling.** Predict the new flower without scaling it and compare the probabilities with the ones above.
3. **Checkpoint the best epoch.** Combine this chapter with Chapter 15's early stopping so that the best model is written to disk whenever validation loss improves.
4. **Count the seeds you need.** Repeat the seed experiment with 30 seeds and see whether the average moves.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What does a `state_dict` contain, and what does it not contain?**
2. **Why does resuming training need the optimizer's state as well as the weights?**
3. **Which three habits must you remember when using a loaded model for prediction?**
4. **Why is a graphics card faster than a processor for training networks?**
5. **Why should the optimizer be created after the model is moved to a device?**
6. **How should you report a result when different seeds give different numbers?**

### Answers

1. The learned numbers, named by layer. It does not contain the network's structure, so you need the code that builds the network to load it.
2. Because optimizers such as Adam keep their own running state for every weight. Without it the resumed run takes different steps; with it, the run here matched the uninterrupted one exactly.
3. Call `model.eval()`, wrap prediction in `torch.no_grad()`, and scale new inputs with the training mean and spread.
4. Because a network's work is mostly huge matrix multiplications made of identical small operations, and a graphics card runs thousands of those at once.
5. Because the optimizer holds references to the weights it updates, so it must be given the weights that actually live on the device.
6. As an average over several seeds with its range, and check that your conclusion holds across them, not just in one lucky run.

## What Comes Next

You can now save a model and a full training checkpoint, resume a run exactly, load a model safely for prediction, move training to a graphics card, and report results that survive a change of seed. That is every tool the final chapter needs.

- **Chapter 20, Your First Deep Learning Project,** uses all of it at once, from choosing the data to presenting a result you can defend.

Chapter 20 begins with a question most tutorials skip: before any network is built, what would count as success?